### Topics Covered (stream processing)
- `medallion architecture`
- `source` and `sinks`
- `pipeline chaining`
- `cleanSource` feature of dirs `readStream` with <i>`delete` and `archive with sourceArchiveDir`</i>

In [0]:
class BronzeLayer:
    def __init__(self, project_dir:str)->None:
        self.project_dir=project_dir
        self.datasetDir="dataset/"
        self.result_df=None
        self.landing_zone_dir="landing_zone/"
        self.checkpointDir="checkpoint/bronze/"
        self.sourceArchiveDir="archive/"
        self.streamName="Bronze Layer Stream"
        self.bronzeLayerTableName="project4_bronze_layer"
        self.queryStream=None
        print("bronze layer initialized !")

    def cleanup_n_setup(self):
        # cleanups
        table_drop_query=f"drop table if exists {self.bronzeLayerTableName}"
        spark.sql(table_drop_query)
        table_data_store_dir=f"/user/hive/warehouse/{self.bronzeLayerTableName}"
        dbutils.fs.rm(table_data_store_dir,True)

        dbutils.fs.rm(self.project_dir+self.checkpointDir, True)
        dbutils.fs.rm(self.project_dir+self.landing_zone_dir, True)
        dbutils.fs.rm(self.project_dir+self.sourceArchiveDir, True)

        # setups
        dbutils.fs.mkdirs(self.project_dir+self.checkpointDir)
        dbutils.fs.mkdirs(self.project_dir+self.landing_zone_dir)
        dbutils.fs.mkdirs(self.project_dir+self.sourceArchiveDir)
        print("BRONZE LAYER CLEANUP AND SETUP COMPLETED !")

    def getTableName(self)->str:
        return self.bronzeLayerTableName

    def ingest_data_to_landing_zone(self, file_name:str):
        """
            `args`: 
                - fileName:str= it is the Name of the file that need to be picked up from dataset dir
            `returns : bool`:
                -   True: if ingestion of file from dataset dir to landing zone completd successfully !
                - False: if there is any error in ingestion process.
        """
        try:
            dbutils.fs.cp(self.project_dir+self.datasetDir+file_name, self.project_dir+self.landing_zone_dir)
            print("BRONZE LAYER DATA INGESTION COMPLETED !", file_name)
        except Exception as e:
            print("BRONZE LAYER DATA INGESTION FAILED ! \nERROR: ", str(e))
            
        
    def get_invoice_schema(self)->str:
        return """ InvoiceNumber string, CreatedTime bigint, StoreID string, PosID string, CashierID string, CustomerType string, CustomerCardNo string, TotalAmount double, SGST double, CESS double, DeliveryType string,
        DeliveryAddress struct< AddressLine string, City string, State string, PinCode string, ContactNumber string>,
        InvoiceLineItems array<struct<ItemCode string, ItemDescription string, ItemPrice double, ItemQty bigint, TotalValue double>> """ 

    def extract_data(self):
        rawDf=spark.readStream.format("json").schema(self.get_invoice_schema()).option("cleanSource","archive").option("sourceArchiveDir",self.project_dir+self.sourceArchiveDir).load(self.project_dir+self.landing_zone_dir+"*.json")
        self.result_df=rawDf
        print("BRONZE LAYER DATA EXTRACTION COMPLETED !")
    
    def load_data(self):
        self.queryStream=self.result_df.writeStream.format("delta").queryName(self.streamName).option("checkpointLocation",self.project_dir+self.checkpointDir).outputMode("append").toTable(self.bronzeLayerTableName)
        print("BRONZE LAYER DATA LOAD COMPLETED !")
        
    
    def stopStreamingQuery(self):
        self.queryStream.stop()
        if self.queryStream.status["message"]=="Stopped":
            print("BRONZE LAYER DATA STREAMING QUERY STOPPED!")
        else:
            print("PROBLEM IN STOPPING BRONZE LAYER DATA STREAMING QUERY!")
    
    def getHeadData(self, count:int):
        return spark.table(self.getTableName()).head(n=count)
    

    


In [0]:
from pyspark.sql.functions import expr, explode, trim, split, lower, upper

In [0]:
class SilverLayer:
    def __init__(self, project_dir:str)->None:
        self.project_dir=project_dir
        self.result_df=None
        self.rawDF=None
        self.checkpointDir="checkpoint/silver/"
        self.streamName="Silver Layer Stream"
        self.bronzeLayerTableName="project4_bronze_layer"
        self.silverLayerTableName="project4_silver_layer"
        self.queryStream=None
        print("Silver layer initialized !")

    def cleanup_n_setup(self):
        # cleanups
        table_drop_query=f"drop table if exists {self.silverLayerTableName}"
        spark.sql(table_drop_query)
        table_data_store_dir=f"/user/hive/warehouse/{self.silverLayerTableName}"
        dbutils.fs.rm(table_data_store_dir,True)

        dbutils.fs.rm(self.project_dir+self.checkpointDir, True)

        # setups
        dbutils.fs.mkdirs(self.project_dir+self.checkpointDir)
        print("SILVER LAYER CLEANUP AND SETUP COMPLETED !")

    def getTableName(self)->str:
        return self.silverLayerTableName
    
    def extract_data(self):
        self.rawDF=spark.readStream.table(self.bronzeLayerTableName)
        print("SILVER LAYER DATA EXTRACTION COMPLETED !")
    
    

    def transform_data(self):
        exploded_df=self.rawDF.selectExpr("InvoiceNumber","CreatedTime","StoreID","PosID","CashierID","CustomerType","CustomerCardNo" ,"TotalAmount" ,"SGST","CESS","DeliveryType",
        "DeliveryAddress.AddressLine", "DeliveryAddress.City","DeliveryAddress.State", "DeliveryAddress.PinCode" , "DeliveryAddress.ContactNumber",
        "explode(InvoiceLineItems) as LineItems")

        transformedDf=exploded_df\
            .withColumn("ItemCode",expr("LineItems.ItemCode"))\
                .withColumn("ItemDescription",expr("LineItems.ItemDescription"))\
                    .withColumn("ItemPrice", expr("LineItems.ItemPrice"))\
                        .withColumn("ItemQty",expr("LineItems.ItemQty"))\
                            .withColumn("TotalValue",expr("LineItems.TotalValue"))\
                                .drop("LineItems")


        self.result_df=transformedDf
        print("SILVER LAYER DATA TRANSFORMATION COMPLETED !")
    

    def load_data(self):
        self.queryStream=self.result_df.writeStream\
            .format("delta")\
                .queryName(self.streamName)\
                    .option("checkpointLocation",self.project_dir+self.checkpointDir)\
                        .outputMode("append")\
                            .toTable(self.silverLayerTableName)
        print("SILVER LAYER DATA LOAD COMPLETED !")

    def getHeadData(self, count:int):
        return spark.table(self.getTableName()).head(n=count)
    
    def stopStream(self)->bool:
        self.queryStream.stop()
        if self.queryStream.status["message"]=="Stopped":
            print("SILVER LAYER DATA STREAMING QUERY STOPPED!")
        else:
            print("PROBLEM IN STOPPING SILVER LAYER DATA STREAMING QUERY!")
    
    
    

### project 4 completed